# Stage 3 — Mid-Level Concatenation Fusion

In this stage, we fuse all three modalities (tabular features, signal features, and text embeddings) into a wide feature panel. We train a LightGBM classifier with synthetic modality dropout on the training folds and average their predictions to predict the target probability for the test set.

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import lightgbm as lgb
from src.data import load_split
from src.cv import get_folds, assert_disjoint_groups
from src.metrics import auprc, report_folds
from src.feature_panel import build_feature_panel
from src.modality_dropout import apply_modality_dropout

RANDOM_STATE = 42

## 1. Load Data Splits
We load both train and test splits.

In [4]:
train = load_split("train")
tab, notes, signals, channel_names = train["tab"], train["notes"], train["signals"], train["channel_names"]
note_text = notes["maintenance_note"]
y = tab["failure_within_horizon"].values
print(f"Train split: {len(tab)} flights, {tab[''drone_id''].nunique()} drones, base rate = {y.mean():.4f}")

test = load_split("test")
tab_test, notes_test, signals_test = test["tab"], test["notes"], test["signals"]
note_text_test = notes_test["maintenance_note"]
print(f"Test split: {len(tab_test)} flights, {tab_test[''drone_id''].nunique()} drones")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1620979010.py, line 5)

## 2. Build Clean Feature Panels
We build clean feature panels for both train and test splits by combining tabular, signal, and text embedding features.

In [ ]:
panel_clean, cat_cols, text_cols = build_feature_panel(tab, note_text, signals, channel_names)
feature_cols = [c for c in panel_clean.columns if c != "flight_id"]
print("Train panel clean shape:", panel_clean.shape)

panel_test, _, _ = build_feature_panel(tab_test, note_text_test, signals_test, channel_names)
print("Test panel shape:", panel_test.shape)

## 3. Train Fusion Model with Modality Dropout
We use the Stratified Group K-Fold scheme. On each fold, we apply synthetic modality dropout (at 15% rate for signal and text modalities) to train a robust LightGBM model. Out-of-fold predictions are collected to evaluate validation AUPRC.

In [ ]:
folds = list(get_folds(tab, n_splits=5, seed=RANDOM_STATE))

lgb_params = dict(
    objective="binary",
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbosity=-1,
)

oof_preds = np.zeros(len(tab))
test_preds_folds = []
fold_scores = []

for fold_i, (tr_idx, val_idx) in enumerate(folds):
    assert_disjoint_groups(tab, tr_idx, val_idx)
    
    # Apply modality dropout to train indices
    sig_aug, txt_aug, _, _ = apply_modality_dropout(
        signals, note_text, tr_idx, p_signal=0.15, p_text=0.15, seed=RANDOM_STATE + fold_i
    )
    
    # Build feature panel for augmented train split
    panel_aug, _, _ = build_feature_panel(tab, txt_aug, sig_aug, channel_names)
    
    # Train model
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        panel_aug.loc[tr_idx, feature_cols], 
        y[tr_idx], 
        categorical_feature=cat_cols
    )
    
    # Evaluate on validation fold (using clean panel)
    val_pred = model.predict_proba(panel_clean.loc[val_idx, feature_cols])[:, 1]
    oof_preds[val_idx] = val_pred
    score = auprc(y[val_idx], val_pred)
    fold_scores.append(score)
    print(f"Fold {fold_i} Validation AUPRC: {score:.5f}")
    
    # Predict on test
    test_pred = model.predict_proba(panel_test[feature_cols])[:, 1]
    test_preds_folds.append(test_pred)

## 4. Evaluation Summary
We look at the fold scores and pool the out-of-fold predictions to evaluate overall model performance.

In [ ]:
report_folds(fold_scores, "Fusion Model (Modality Dropout)")
pooled_oof_score = auprc(y, oof_preds)
print(f"Pooled OOF AUPRC: {pooled_oof_score:.5f}")

## 5. Generate and Verify final Test Predictions
We average the predictions from the 5 fold models and export them to `submission.csv` in the format required by the submission guidelines.

In [ ]:
final_test_preds = np.mean(test_preds_folds, axis=0)

submission = pd.DataFrame({
    "flight_id": tab_test["flight_id"],
    "failure_within_horizon": final_test_preds
})

submission.to_csv("../submission.csv", index=False)
print("Saved submission to ../submission.csv")

# Verify submission.csv properties
import os
sub_file = "../submission.csv"
assert os.path.exists(sub_file), "submission.csv was not created!"
lines = open(sub_file).readlines()
print(f"Number of lines in submission.csv: {len(lines)}")
assert len(lines) == len(tab_test) + 1, "Line count mismatch!"
print(submission["failure_within_horizon"].describe())